# Cleaning and analysis of the attendance dataset

This notebook is more about illustrating the process of thinking, rather than doing the analysis the correct way or presenting some results. When we start working with a new dataset, we should clean it and inspect its quality. What we discover during this process, and the decisions we make while cleaning the dataset will have impact on the rest of the analysis.

You can find the documentation for this dataset [here](https://github.com/harismck/ism-data-science-2026/blob/main/docs/attendance.md).

1. 


In [2]:
import duckdb

db_path = "data.duckdb"
schema = "data.attendance"

conn = duckdb.connect()
conn.execute(f"ATTACH '{db_path}' AS data (READ_ONLY)")
conn.execute(f"USE {schema}");


# Week 1 - Data Cleaning with SQL and duckdb

Let's first extract a clean dataset that we can use in further analysis confidently.


The first step when facing some new data is to look at it. This is something that applies always, regardless of what problem you're facing - look at your data. Let's look at each table in turn.

Here are some best practices:
- It's always a good idea to look at a random sample of the data, rather than the first/last N rows.
- Begin with small tables first (in our case, `school` and `subject`) - tables that are about a single thing - rather than bigger tables that are a combination of multiple things.

## school

In [3]:
conn.sql(
    "SELECT * FROM school ORDER BY random() LIMIT 10"
).show()

┌─────────────────┬─────────────┬───────────────┬───────────────────────────────────────────────────────────────────┬───────────────┬───────────────────┬───────────────────┐
│    unique_id    │ school_code │ division_code │                            school_name                            │ division_name │ municipality_code │ municipality_name │
│     varchar     │   varchar   │    varchar    │                              varchar                              │    varchar    │      varchar      │      varchar      │
├─────────────────┼─────────────┼───────────────┼───────────────────────────────────────────────────────────────────┼───────────────┼───────────────────┼───────────────────┤
│ 1a2a6262bd45dfc │ 190453670   │ NULL          │ Marijampolės „Ryto“ progimnazija                                  │ NULL          │ 18                │ Marijampolės sav. │
│ 6938a0195be3787 │ 290140580   │ NULL          │ Kauno Motiejaus Valančiaus mokykla-darželis                       │ NULL        

`school_code` should be the primary key in this table. Primary key is a column (or several columns) by which a unique row in the table is identified. By stating that `school_code` is a primary key, I explicitly expect two things from the column:
- This column has no duplicate values.
- This column has no null values.

Let's verify these assumptions.

In [4]:
conn.sql(
    """
    SELECT 
        school_code,
        COUNT(*)
    FROM school
    GROUP BY school_code
    HAVING COUNT(*) > 1
"""
).show()

┌─────────────┬──────────────┐
│ school_code │ count_star() │
│   varchar   │    int64     │
├─────────────┼──────────────┤
│ 290893610   │            2 │
│ 190066944   │            3 │
│ 195170434   │            2 │
│ 190189861   │            2 │
│ 191090275   │            2 │
│ 190696786   │            2 │
│ 190697735   │            2 │
│ 302843970   │            2 │
│ 300556726   │            2 │
│ 190341625   │            2 │
│ 191130079   │            3 │
│ 190892856   │            2 │
│ 191791956   │            2 │
│ 190341810   │            2 │
│ 190545880   │            3 │
│ 190057176   │            2 │
│ 191056433   │            2 │
│ 190986889   │            2 │
│ 191130983   │            3 │
│ 295093070   │            2 │
├─────────────┴──────────────┤
│ 20 rows          2 columns │
└────────────────────────────┘



We have duplicates :/ Let's look at a couple of them.

In [5]:
conn.sql(
    """
    SELECT *
    FROM school
    WHERE school_code IN (190341810, 191130983, 191130079)
"""
).show()

┌─────────────────┬─────────────┬───────────────┬─────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────┬───────────────────┬───────────────────┐
│    unique_id    │ school_code │ division_code │                 school_name                 │                            division_name                            │ municipality_code │ municipality_name │
│     varchar     │   varchar   │    varchar    │                   varchar                   │                               varchar                               │      varchar      │      varchar      │
├─────────────────┼─────────────┼───────────────┼─────────────────────────────────────────────┼─────────────────────────────────────────────────────────────────────┼───────────────────┼───────────────────┤
│ 171ce47c9346344 │ 191130983   │ 7403          │ Plungės r. Žemaitijos kadetų gimnazija      │ Plungės r. Žemaitijos kadetų Kulių gimnazijos skyrius               │ 68        

Looks like the school table is actually unique by school_code and division_code, but let's verify again

In [6]:
conn.sql(
    """
    SELECT 
        school_code,
        division_code,
        COUNT(*)
    FROM school 
    GROUP BY school_code, division_code
    HAVING COUNT(*) > 1
"""
).show()

┌─────────────┬───────────────┬──────────────┐
│ school_code │ division_code │ count_star() │
│   varchar   │    varchar    │    int64     │
├─────────────┴───────────────┴──────────────┤
│                   0 rows                   │
└────────────────────────────────────────────┘



This is a data quality issue - it's unclear what division code NULL means, and the table documentation makes it seems that this is data on schools and not school divisions.

I'll deal with it in a somewhat silly way - I'll remove cases where `division_code` is not null. This is essentially saying - I won't look at schools that have divisions. Ideally we wouldn't do this, but out of concern for time, let's make this simplification. Also only 17 `school_codes` have the duplicate issue.

Simplifying in the beginning of an analysis is a good approach to manage complexity. We still don't know the dataset very well, so let's simplify. Once we are more familiar with the data, we can increase complexity and make our analysis more "correct".

Let's check if there are no null `school_codes`.


In [15]:
conn.sql(
    """
    SELECT 
        count(distinct school_code),
        count(*)
    FROM school
"""
)

┌─────────────────────────────┬──────────────┐
│ count(DISTINCT school_code) │ count_star() │
│            int64            │    int64     │
├─────────────────────────────┼──────────────┤
│                        1029 │         1053 │
└─────────────────────────────┴──────────────┘

In [17]:
# Final query for the school table
school = """
    SELECT 
        school_code, 
        school_name, 
        municipality_name
    FROM school 
    WHERE division_code IS NULL
"""
conn.sql(school + " LIMIT 10").show()

┌─────────────┬─────────────────────────────────────────────────┬─────────────────────┐
│ school_code │                   school_name                   │  municipality_name  │
│   varchar   │                     varchar                     │       varchar       │
├─────────────┼─────────────────────────────────────────────────┼─────────────────────┤
│ 190545880   │ Biržų „Aušros“ pagrindinė mokykla               │ Biržų r. sav.       │
│ 191130264   │ Plungės „Saulės“ gimnazija                      │ Plungės r. sav.     │
│ 191814839   │ Mažeikių „Žiburėlio“ pradinė mokykla            │ Mažeikių r. sav.    │
│ 190507118   │ Švenčionių r. Adutiškio pagrindinė mokykla      │ Švenčionių r. sav.  │
│ 191317260   │ Vilniaus Trakų Vokės gimnazija                  │ Vilniaus m. sav.    │
│ 191709681   │ Vilniaus Jono Ivaškevičiaus jaunimo mokykla     │ Vilniaus m. sav.    │
│ 290486810   │ Vilkaviškio rajono Pajevonio pagrindinė mokykla │ Vilkaviškio r. sav. │
│ 303283300   │ Panevėžio Raimun

## subject


In [18]:
conn.sql(
    "SELECT * FROM subject ORDER BY random() LIMIT 10"
).show()

┌─────────────────┬──────────────────┬──────────────┬─────────────────────────────────────┬───────────────────────────────────┐
│    unique_id    │ electronic_diary │ subject_code │            subject_name             │          subject_name_en          │
│     varchar     │     varchar      │   varchar    │               varchar               │              varchar              │
├─────────────────┼──────────────────┼──────────────┼─────────────────────────────────────┼───────────────────────────────────┤
│ 8d8ca62b82f82c9 │ Tamo             │ 4100         │ Lietuvių kalba                      │ Lithuanian language               │
│ a47f434d018c4de │ NASC             │ 21014        │ Keramika                            │ Ceramics                          │
│ b950652313c2ace │ Tamo             │ 10105        │ Taikomasis menas, amatai, dizainas  │ Applied art, crafts, design       │
│ 431a6f0172bb93a │ Tamo             │ 9108         │ Kompiuterinės muzikos technologijos │ Computer mus

This table should be unique by `electronic_diary` and `subject_code`, let's check.


In [19]:
conn.sql(
    """
    SELECT
        electronic_diary,
        subject_code,
        COUNT(*)
    FROM subject
    GROUP BY all
    HAVING COUNT(*) > 1
"""
).show()

┌──────────────────┬──────────────┬──────────────┐
│ electronic_diary │ subject_code │ count_star() │
│     varchar      │   varchar    │    int64     │
├──────────────────┴──────────────┴──────────────┤
│                     0 rows                     │
└────────────────────────────────────────────────┘



In [20]:
# Final query for the subject table
subject = """
    SELECT
        electronic_diary,
        subject_code,
        subject_name_en,
    FROM subject
"""

yay!

## attendance

In [21]:
conn.sql(
    "SELECT * FROM attendance ORDER BY random() LIMIT 10"
).show()

┌─────────────────┬───────────────┬─────────────┬───────────────┬───────────────┬──────────────┬──────────────────┬───────────────────┬─────────────────────┬─────────────────────────┬───────────────────────┬───────────────────┐
│    unique_id    │ report_period │ school_code │ division_code │ student_class │ subject_code │ electronic_diary │ student_count_nsa │ student_count_diary │ excused_lessons_illness │ excused_lessons_other │ unexcused_lessons │
│     varchar     │    varchar    │   varchar   │    varchar    │    varchar    │   varchar    │     varchar      │      varchar      │       varchar       │         varchar         │        varchar        │      varchar      │
├─────────────────┼───────────────┼─────────────┼───────────────┼───────────────┼──────────────┼──────────────────┼───────────────────┼─────────────────────┼─────────────────────────┼───────────────────────┼───────────────────┤
│ 60af2a39f578c4b │ 2020-09-01    │ 191777611   │ NULL          │ 9             │ 9101  

In [22]:
conn.sql("SELECT count(*) FROM attendance")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      7101363 │
└──────────────┘

- The rows in this table should be unique by `report_period`, `school_code`, `student_class` and `subject_code`. I should add `division_code` to this list, but I've already decided to not use that column. `electronic_diary` is not in this list because, if I remember correctly, each school uses just one of the diaries.
- `school_code` is the foreign key from the school table, while `subject_code` and `electronic_diary` are foreign keys, therefore I expect them to not have any NULL values.

Let's verify.


In [23]:
conn.sql(
    """
    SELECT 
        report_period,
        school_code,
        student_class,
        subject_code,
        COUNT(*)
    FROM attendance
    WHERE division_code IS NULL
    GROUP BY all
    HAVING COUNT(*) > 1
"""
).show()

┌───────────────┬─────────────┬───────────────┬──────────────┬──────────────┐
│ report_period │ school_code │ student_class │ subject_code │ count_star() │
│    varchar    │   varchar   │    varchar    │   varchar    │    int64     │
├───────────────┼─────────────┼───────────────┼──────────────┼──────────────┤
│ 2023-10-01    │ 190673983   │ 8             │ 8101         │            2 │
│ 2024-10-01    │ 190057219   │ 5             │ 6001         │            2 │
│ 2024-12-01    │ 190002226   │ 5             │ 7104         │            2 │
│ 2025-03-01    │ 191024624   │ 10            │ 05301        │            2 │
│ 2024-03-01    │ 195092544   │ 8             │ 8201         │            2 │
│ 2025-06-01    │ 190422397   │ 6             │ 4001         │            2 │
│ 2019-11-01    │ 190022976   │ 2             │ 1            │            2 │
│ 2024-06-01    │ 190983430   │ 1             │ 9102         │            2 │
│ 2023-10-01    │ 190342150   │ 5             │ 6001         │  

We have duplicates in the dataset again. But how many duplicates?

In [24]:
conn.sql(
    """
    SELECT 
        report_period,
        school_code,
        student_class,
        subject_code,
        COUNT(*)
    FROM attendance
    WHERE division_code IS NULL
    GROUP BY all
    HAVING COUNT(*) > 1
    ORDER BY random()
"""
).df().shape

(15350, 5)

Quite a few duplicated rows. 

Maybe I should have also included `electronic_diary` among the columns by which the attendance table is unique? This is weird, because why would a single class in a single school use two diaries? But weird things happen.

In [25]:
conn.sql(
    """
    SELECT 
        report_period,
        school_code,
        division_code,
        student_class,
        subject_code,
        electronic_diary,
        COUNT(*)
    FROM attendance
    WHERE division_code IS NULL
    GROUP BY all
    HAVING COUNT(*) > 1
    ORDER BY random()
"""
).df().shape

(11425, 7)

Ok, even grouping by `electronic_diary`, we have duplicates in the dataset. Let's look at several duplicated rows again.

In [26]:
conn.sql(
    """
    SELECT 
        report_period,
        school_code,
        division_code,
        student_class,
        subject_code,
        electronic_diary,
        COUNT(*)
    FROM attendance
    WHERE division_code IS NULL
    GROUP BY all
    HAVING COUNT(*) > 1
    ORDER BY random()
    LIMIT 5
"""
).show()

┌───────────────┬─────────────┬───────────────┬───────────────┬──────────────┬──────────────────┬──────────────┐
│ report_period │ school_code │ division_code │ student_class │ subject_code │ electronic_diary │ count_star() │
│    varchar    │   varchar   │    varchar    │    varchar    │   varchar    │     varchar      │    int64     │
├───────────────┼─────────────┼───────────────┼───────────────┼──────────────┼──────────────────┼──────────────┤
│ 2025-02-01    │ 195472849   │ NULL          │ 8             │ 4001         │ NASC             │            2 │
│ 2024-06-01    │ 191722390   │ NULL          │ 10            │ 7101         │ NASC             │            2 │
│ 2023-10-01    │ 190189676   │ NULL          │ 11            │ 8301         │ NASC             │            2 │
│ 2023-11-01    │ 191651922   │ NULL          │ 11            │ 8101         │ NASC             │            2 │
│ 2024-01-01    │ 190160653   │ NULL          │ 7             │ 6001         │ NASC             

In [27]:
# Here I'm taking the first row from above and inserting the required values
conn.sql(
    """
    SELECT
        *
    FROM attendance
    WHERE division_code IS NULL
    AND school_code = 290984490
    AND student_class = 1
    AND subject_code = 9101
    ORDER BY report_period
"""
)

┌─────────────────┬───────────────┬─────────────┬───────────────┬───────────────┬──────────────┬──────────────────┬───────────────────┬─────────────────────┬─────────────────────────┬───────────────────────┬───────────────────┐
│    unique_id    │ report_period │ school_code │ division_code │ student_class │ subject_code │ electronic_diary │ student_count_nsa │ student_count_diary │ excused_lessons_illness │ excused_lessons_other │ unexcused_lessons │
│     varchar     │    varchar    │   varchar   │    varchar    │    varchar    │   varchar    │     varchar      │      varchar      │       varchar       │         varchar         │        varchar        │      varchar      │
├─────────────────┼───────────────┼─────────────┼───────────────┼───────────────┼──────────────┼──────────────────┼───────────────────┼─────────────────────┼─────────────────────────┼───────────────────────┼───────────────────┤
│ 047c0a11c721ffe │ 2015-11-01    │ 290984490   │ NULL          │ 1             │ 9101  

It seems that in this case we just have duplicated rows. Here's what I'm noticing:
- for the duplicated rows, values in columns `student_count_diary` and `student_count_nsa` are the same.
- Values for the other columns (`excused_lessons_illness` etc.) are usually different. 

To solve the problem of duplicates in this table, I'll just aggregate by the columns that I think should uniquely identify an observation in this dataset.

In [35]:
attendance_final = """
    SELECT
        report_period,
        school_code,
        student_class,
        subject_code,
        electronic_diary,
        FIRST(student_count_diary) student_count_diary,
        SUM(excused_lessons_illness) excused_lessons_illness,
        SUM(excused_lessons_other) excused_lessons_other,
        SUM(unexcused_lessons) unexcused_lessons
    FROM attendance
    WHERE division_code IS NULL
    GROUP BY all
"""
conn.sql(attendance_final + " LIMIT 10").show()

BinderException: Binder Error: No function matches the given name and argument types 'sum(VARCHAR)'. You might need to add explicit type casts.
	Candidate functions:
	sum(DECIMAL) -> DECIMAL
	sum(BOOLEAN) -> HUGEINT
	sum(SMALLINT) -> HUGEINT
	sum(INTEGER) -> HUGEINT
	sum(BIGINT) -> HUGEINT
	sum(HUGEINT) -> HUGEINT
	sum(DOUBLE) -> DOUBLE
	sum(BIGNUM) -> BIGNUM


This error is saying that you cannot sum a variable that has string datatype. It seems some of the lessons count variables are not encoded as integers in the dataset, let's fix that.

In [36]:
attendance_final = """
    SELECT
        report_period,
        school_code,
        student_class,
        subject_code,
        electronic_diary,
        FIRST(student_count_diary) student_count_diary,
        SUM(excused_lessons_illness::int) excused_lessons_illness,
        SUM(excused_lessons_other::int) excused_lessons_other,
        SUM(unexcused_lessons::int) unexcused_lessons
    FROM attendance
    WHERE division_code IS NULL
    GROUP BY all
"""
conn.sql(attendance_final + " LIMIT 10").show()

┌───────────────┬─────────────┬───────────────┬──────────────┬──────────────────┬─────────────────────┬─────────────────────────┬───────────────────────┬───────────────────┐
│ report_period │ school_code │ student_class │ subject_code │ electronic_diary │ student_count_diary │ excused_lessons_illness │ excused_lessons_other │ unexcused_lessons │
│    varchar    │   varchar   │    varchar    │   varchar    │     varchar      │       varchar       │         int128          │        int128         │      int128       │
├───────────────┼─────────────┼───────────────┼──────────────┼──────────────────┼─────────────────────┼─────────────────────────┼───────────────────────┼───────────────────┤
│ 2023-09-01    │ 190595189   │ 8             │ 06001        │ Tamo             │ 12                  │                      24 │                     0 │                 2 │
│ 2022-04-01    │ 190003851   │ 6             │ 10100        │ Tamo             │ 139                 │                      41 │ 

I now have a dataset that I am sure is unique by `report_period`, `school_code`, `student_class` and `subject_code`, `electronic_diary` because that's how GROUP BY works.


In [38]:
conn.sql("""
    SELECT * FROM attendance
    WHERE report_period = '2019-01-01'
    AND school_code = 190001124
    AND student_class = 12
    AND subject_code = 8101
"""
).show()


┌─────────────────┬───────────────┬─────────────┬───────────────┬───────────────┬──────────────┬──────────────────┬───────────────────┬─────────────────────┬─────────────────────────┬───────────────────────┬───────────────────┐
│    unique_id    │ report_period │ school_code │ division_code │ student_class │ subject_code │ electronic_diary │ student_count_nsa │ student_count_diary │ excused_lessons_illness │ excused_lessons_other │ unexcused_lessons │
│     varchar     │    varchar    │   varchar   │    varchar    │    varchar    │   varchar    │     varchar      │      varchar      │       varchar       │         varchar         │        varchar        │      varchar      │
├─────────────────┼───────────────┼─────────────┼───────────────┼───────────────┼──────────────┼──────────────────┼───────────────────┼─────────────────────┼─────────────────────────┼───────────────────────┼───────────────────┤
│ 1f07da06f4a2fe0 │ 2019-01-01    │ 190001124   │ NULL          │ 12            │ 8101  

## final query

Ok, I think I'm ready to work on the final dataset now, so let's join everything together. I saved my queries into Python variables for a reason - now combining them is very easy. 

Keep in mind, that below I'm only using the WITH statement and inserting the tables separately because to all of them except the `subject` table I had to apply some sort of deduplication or grouping - logic that I now want to end up in the final query as well.

Also note that we're using GROUP BY to get rid of the `electronic_diary` column. There are some rows that have the same `school`, `subject`, `period` and `student_class`, but different `electronic_diary` values. I'm not sure what explains this, maybe it's schools transitioning from one diary to another. For my analysis, however, having the `electronic_diary` column is not important, therefore let's get rid of it.

Also note how big the final query is - that's a lot of logic! It would have been very tricky to such a query from the get go. That's why we went through this iterative process in the seminar.


In [31]:
attendance_query = """
    SELECT
        report_period,
        school_code,
        student_class,
        subject_code,
        electronic_diary,
        FIRST(student_count_diary) student_count_diary,
        SUM(excused_lessons_illness::int) excused_lessons_illness,
        SUM(excused_lessons_other::int) excused_lessons_other,
        SUM(unexcused_lessons::int) unexcused_lessons
    FROM attendance
    WHERE division_code IS NULL
    GROUP BY all
"""

subject_query = """
    SELECT
        electronic_diary,
        subject_code,
        subject_name_en,
    FROM subject
"""

school_query = """
    SELECT 
        school_code, 
        school_name, 
        municipality_name
    FROM school 
    WHERE division_code IS NULL
"""


query = f"""
WITH attendance_clean AS ({attendance_query}),
    school_clean AS ({school_query}),
    subject_clean AS ({subject_query})
SELECT
    school_clean.school_name,
    school_clean.school_code,
    school_clean.municipality_name,
    attendance_clean.student_class,
    attendance_clean.report_period,
    subject_clean.subject_name_en,
    FIRST(attendance_clean.student_count_diary) AS student_count,
    SUM(attendance_clean.excused_lessons_illness) AS excused_lessons_illness,
    SUM(attendance_clean.excused_lessons_other) AS excused_lessons_other,
    SUM(attendance_clean.unexcused_lessons) AS unexcused_lessons
FROM attendance_clean
JOIN school_clean ON attendance_clean.school_code = school_clean.school_code
JOIN subject_clean ON attendance_clean.subject_code = subject_clean.subject_code
    AND attendance_clean.electronic_diary = subject_clean.electronic_diary
GROUP BY all
"""
print(query)


WITH attendance_clean AS (
    SELECT
        report_period,
        school_code,
        student_class,
        subject_code,
        electronic_diary,
        FIRST(student_count_diary) student_count_diary,
        SUM(excused_lessons_illness::int) excused_lessons_illness,
        SUM(excused_lessons_other::int) excused_lessons_other,
        SUM(unexcused_lessons::int) unexcused_lessons
    FROM attendance
    WHERE division_code IS NULL
    GROUP BY all
),
    school_clean AS (
    SELECT 
        school_code, 
        school_name, 
        municipality_name
    FROM school 
    WHERE division_code IS NULL
),
    subject_clean AS (
    SELECT
        electronic_diary,
        subject_code,
        subject_name_en,
    FROM subject
)
SELECT
    school_clean.school_name,
    school_clean.school_code,
    school_clean.municipality_name,
    attendance_clean.student_class,
    attendance_clean.report_period,
    subject_clean.subject_name_en,
    FIRST(attendance_clean.student_count_d

In [32]:
conn.sql(query).df()


,school_name,school_code,municipality_name,student_class,report_period,subject_name_en,student_count,excused_lessons_illness,excused_lessons_other,unexcused_lessons
0,Kauno Kazio Griniaus progimnazija,190138938,Kauno m. sav.,8,2019-12-01,Human safety,143,16.0,3.0,4.0
1,Druskininkų savivaldybės Leipalingio progimnazija,190609055,Druskininkų sav.,1,2024-11-01,Technologies,8,5.0,0.0,0.0
2,Utenos Adolfo Šapokos gimnazija,193295784,Utenos r. sav.,10,2024-10-01,Physical education,119,34.0,5.0,5.0
3,Kaišiadorių šventosios Faustinos ugdymo centras,190984870,Kaišiadorių r. sav.,7,2025-04-01,... (elective),3,0.0,0.0,2.0
4,Kauno r. Vilkijos gimnazija,191089910,Kauno r. sav.,1,2021-09-01,Dance,11,4.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
3616213,Panevėžio Vytauto Mikalausko menų gimnazija,300594538,Panevėžio m. sav.,10,2022-01-01,... (elective),28,19.0,0.0,0.0
3616214,Klaipėdos Vydūno gimnazija,190910382,Klaipėdos m. sav.,6,2022-05-01,Performer's expression (choir singing and cond...,88,5.0,10.0,0.0
3616215,Kauno r. Zapyškio pagrindinė mokykla,191092511,Kauno r. sav.,2,2022-03-01,... (elective),9,22.0,0.0,0.0
3616216,Vilniaus r. Marijampolio Meilės Lukšienės gimn...,191339495,Vilniaus r. sav.,10,2024-05-01,... (subject module),13,3.0,2.0,5.0
